In [ ]:
import os
import json
import math
import numpy as np
import joblib
import pandas as pd
from datetime import timedelta

# TensorFlow / Keras
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.preprocessing import MinMaxScaler
from typing import Tuple, Dict, Any
from math import sqrt

# Reproducibility
def set_global_seed(seed: int = 42):
    import random
    random.seed(seed)
    np.random.seed(seed)
    tf.random.set_seed(seed)

set_global_seed(42)

print(f"TensorFlow Version: {tf.__version__}")

In [ ]:
# === Configuration ===
CSV_PATH = "./dataset/merged_stock_data.csv"

# Columns
DATE_COLUMN = "Date"
SCRIP_COLUMN = "Scrip"
TARGET_COLUMN = "Close"

# Using ALL available features for multivariate recursion
FEATURE_COLS = ["Close", "Open", "High", "Low", "Volume", "SMA_14", "RSI_14"]

# Model Params
SEQ_LEN = 60
# We only train for 1 step ahead, but we evaluate on these horizons recursively
EVAL_HORIZONS = [1, 3, 7, 15, 30]

# Training Params
TRAIN_RATIO = 0.8
MAX_EPOCHS = 100
BATCH_SIZE = 64
LEARNING_RATE = 1e-3
PATIENCE = 10
VAL_SPLIT = 0.2

# Save Paths
SAVE_DIR = "./artifacts_recursive"
MODELS_DIR = os.path.join(SAVE_DIR, "models")

os.makedirs(MODELS_DIR, exist_ok=True)
print(f"Saving models to: {os.path.abspath(MODELS_DIR)}")

In [ ]:
# === Data Loading & Feature Engineering ===

def add_technical_indicators(df: pd.DataFrame) -> pd.DataFrame:
    # SMA 14
    df['SMA_14'] = df.groupby(SCRIP_COLUMN)[TARGET_COLUMN].transform(lambda x: x.rolling(window=14).mean())

    # RSI 14
    def calculate_rsi(x, window=14):
        delta = x.diff()
        gain = (delta.where(delta > 0, 0)).rolling(window=window).mean()
        loss = (-delta.where(delta < 0, 0)).rolling(window=window).mean()
        rs = gain / loss
        return 100 - (100 / (1 + rs))

    df['RSI_14'] = df.groupby(SCRIP_COLUMN)[TARGET_COLUMN].transform(lambda x: calculate_rsi(x))

    # Fill NaNs
    df = df.dropna().reset_index(drop=True)
    return df

def read_all_data(csv_path: str) -> pd.DataFrame:
    df = pd.read_csv(csv_path)
    # Ensure types and sorting
    df[DATE_COLUMN] = pd.to_datetime(df[DATE_COLUMN])
    df = df.sort_values([SCRIP_COLUMN, DATE_COLUMN]).reset_index(drop=True)

    # Add indicators
    df = add_technical_indicators(df)
    return df

def get_scrips(df: pd.DataFrame) -> np.ndarray:
    return df[SCRIP_COLUMN].dropna().unique()

In [ ]:
# === Preprocessing ===

def split_train_test(values_len: int, train_ratio: float) -> Tuple[int, int]:
    train_end = int(values_len * train_ratio)
    test_start = train_end
    return train_end, test_start

def fit_scaler_on_train(train_df: pd.DataFrame) -> MinMaxScaler:
    # LSTM default activation is tanh (-1 to 1). Matching input scale helps convergence.
    scaler = MinMaxScaler(feature_range=(-1, 1))
    scaler.fit(train_df[FEATURE_COLS].values)
    return scaler

def scale_df(df: pd.DataFrame, scaler: MinMaxScaler) -> np.ndarray:
    return scaler.transform(df[FEATURE_COLS].values)

def build_sequences_recursive(values: np.ndarray, seq_len: int) -> Tuple[np.ndarray, np.ndarray]:
    """
    Creates (X, y) pairs where:
    X = sequence of length seq_len (all features)
    y = the NEXT single step (all features)
    """
    X, y = [], []
    T = len(values)
    # We need at least seq_len + 1 data points
    for i in range(T - seq_len):
        window = values[i : i + seq_len]
        target = values[i + seq_len] # Predict the very next row (all features)
        X.append(window)
        y.append(target)
    return np.array(X), np.array(y)

def prepare_scrip_data(scrip_df: pd.DataFrame, seq_len: int, train_ratio: float) -> Dict[str, Any]:
    n = len(scrip_df)
    train_end, test_start = split_train_test(n, train_ratio)

    train_df = scrip_df.iloc[:train_end].copy()
    test_df  = scrip_df.iloc[test_start:].copy()

    # Fit scaler on TRAIN only
    scaler = fit_scaler_on_train(train_df)

    # Transform
    train_scaled = scale_df(train_df, scaler)
    test_scaled  = scale_df(test_df, scaler)

    # Build sequences (1-step ahead target)
    X_train, y_train = build_sequences_recursive(train_scaled, seq_len)
    X_test, y_test   = build_sequences_recursive(test_scaled, seq_len)

    return {
        "scaler": scaler,
        "train_df": train_df,
        "test_df": test_df,
        "X_train": X_train, "y_train": y_train,
        "X_test": X_test, "y_test": y_test,
        "test_scaled": test_scaled # Needed for recursive evaluation loop
    }

In [ ]:
# === Model Definition ===

def build_lstm_model(input_shape, n_features: int, lr: float = LEARNING_RATE) -> keras.Model:
    model = keras.Sequential([
        layers.Input(shape=input_shape),
        # Bidirectional allows learning from both past-to-future and future-to-past context in the window
        layers.Bidirectional(layers.LSTM(64, return_sequences=True)),
        layers.Dropout(0.2),
        layers.Bidirectional(layers.LSTM(32)),
        layers.Dropout(0.2),
        # Output layer predicts ALL features for the next step
        layers.Dense(n_features)
    ])

    # Huber loss is less sensitive to outliers than MSE, which is good for noisy stock data
    model.compile(optimizer=keras.optimizers.Adam(learning_rate=lr),
                  loss=keras.losses.Huber(),
                  metrics=["mae"])
    return model

def get_callbacks():
    es = keras.callbacks.EarlyStopping(monitor="val_loss", patience=PATIENCE, restore_best_weights=True)
    rlrop = keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=max(3, PATIENCE//2), min_lr=1e-5)
    return [es, rlrop]

In [ ]:
# === Recursive Prediction & Evaluation ===

def recursive_predict(model, initial_sequence: np.ndarray, n_steps: int) -> np.ndarray:
    """
    Predicts n_steps recursively.
    initial_sequence: shape (SEQ_LEN, n_features)
    Returns: shape (n_steps, n_features)
    """
    current_seq = initial_sequence.copy() # (SEQ_LEN, n_features)
    predictions = []

    for _ in range(n_steps):
        # Reshape for model (1, SEQ_LEN, n_features)
        input_tensor = current_seq.reshape(1, *current_seq.shape)

        # Predict next step (1, n_features)
        next_step_pred = model.predict(input_tensor, verbose=0)

        # Store prediction
        predictions.append(next_step_pred[0])

        # Update sequence: remove first, append predicted
        # current_seq is (SEQ_LEN, n_features)
        # next_step_pred is (1, n_features) -> reshape to (1, n_features) if needed, but [0] gave us (n_features,)

        # Shift
        current_seq = np.roll(current_seq, -1, axis=0)
        current_seq[-1] = next_step_pred[0]

    return np.array(predictions)

def evaluate_recursive(model, test_scaled: np.ndarray, scaler: MinMaxScaler, horizons: list) -> Dict[str, float]:
    """
    Evaluates the model using a recursive strategy on the test set.
    We pick a few starting points in the test set to evaluate long-term accuracy.
    For simplicity in this script, we'll evaluate starting from the BEGINNING of the test set
    and maybe a few other points, or just the very last window?

    Standard practice: Rolling origin evaluation.
    Here, we will evaluate on the LAST available window to see how it predicts the "future" (if we had ground truth),
    but since we need ground truth to calculate RMSE, we should pick a point `max(horizons)` steps before the end.
    """

    # We need at least SEQ_LEN + max_horizon data points
    max_h = max(horizons)
    if len(test_scaled) < SEQ_LEN + max_h:
        return {}

    # Let's evaluate on the LAST valid window where we have full ground truth
    # Start index such that start + SEQ_LEN + max_h <= len(test_scaled)
    start_idx = len(test_scaled) - SEQ_LEN - max_h

    # Initial window
    initial_window = test_scaled[start_idx : start_idx + SEQ_LEN]

    # Ground truth for next max_h steps
    ground_truth_scaled = test_scaled[start_idx + SEQ_LEN : start_idx + SEQ_LEN + max_h]

    # Predict recursively
    predictions_scaled = recursive_predict(model, initial_window, max_h)

    # Inverse transform
    # We need to inverse transform the whole (n_steps, n_features) matrix
    predictions_actual = scaler.inverse_transform(predictions_scaled)
    ground_truth_actual = scaler.inverse_transform(ground_truth_scaled)

    # Extract "Close" column (index 0 in FEATURE_COLS)
    close_idx = FEATURE_COLS.index("Close")
    pred_close = predictions_actual[:, close_idx]
    true_close = ground_truth_actual[:, close_idx]

    metrics = {}

    for h in horizons:
        # Slice first h steps
        p = pred_close[:h]
        t = true_close[:h]

        rmse_val = sqrt(np.mean((t - p) ** 2))
        mape_val = np.mean(np.abs((t - p) / np.maximum(np.abs(t), 1e-8))) * 100.0

        metrics[f"rmse_{h}d"] = rmse_val
        metrics[f"mape_{h}d"] = mape_val

    return metrics

In [ ]:
# === Main Training Loop ===

df_all = read_all_data(CSV_PATH)
scrips = get_scrips(df_all)
print(f"Found {len(scrips)} scrips.")

# Filter for specific scrip if needed, or train all
# scrips = ["ACI"]

for idx, scrip in enumerate(scrips):
    print(f"\n[{idx+1}/{len(scrips)}] Processing {scrip}...")

    scrip_df = df_all[df_all[SCRIP_COLUMN] == scrip].sort_values(DATE_COLUMN).reset_index(drop=True)
    scrip_df = scrip_df.dropna(subset=FEATURE_COLS)

    # Check size
    if len(scrip_df) < (SEQ_LEN + max(EVAL_HORIZONS) + 20):
        print(f"  Skipping {scrip}: insufficient data.")
        continue

    # Prepare data
    data = prepare_scrip_data(scrip_df, SEQ_LEN, TRAIN_RATIO)
    X_train, y_train = data["X_train"], data["y_train"]
    X_test, y_test = data["X_test"], data["y_test"]
    scaler = data["scaler"]

    if len(X_train) == 0:
        print("  Skipping: empty train set.")
        continue

    # Build Model
    n_features = len(FEATURE_COLS)
    input_shape = (SEQ_LEN, n_features)
    model = build_lstm_model(input_shape, n_features)

    # Train
    model.fit(
        X_train, y_train,
        epochs=MAX_EPOCHS,
        batch_size=BATCH_SIZE,
        validation_split=VAL_SPLIT,
        verbose=0,
        callbacks=get_callbacks(),
        shuffle=False
    )

    # Save Model & Scaler
    scrip_dir = os.path.join(MODELS_DIR, scrip)
    os.makedirs(scrip_dir, exist_ok=True)

    model_path = os.path.join(scrip_dir, f"lstm_recursive_{scrip}.keras")
    model.save(model_path)

    scaler_path = os.path.join(scrip_dir, f"scaler_{scrip}.bin")
    if not os.path.exists(scaler_path):
        joblib.dump(scaler, scaler_path)

    # Evaluate Recursively
    metrics = evaluate_recursive(model, data["test_scaled"], scaler, EVAL_HORIZONS)

    print(f"  Saved to {model_path}")
    if metrics:
        print("  Recursive Eval (Last Window):")
        for k, v in metrics.items():
            print(f"    {k}: {v:.4f}")
    else:
        print("  Could not evaluate recursive metrics (test set too small).")

print("\nAll training completed.")

In [ ]:
# === Example: Load and Predict Future ===

EXAMPLE_SCRIP = "ACI" # Change to a valid scrip
PREDICT_DAYS = 15

if EXAMPLE_SCRIP in scrips:
    print(f"\nGenerating {PREDICT_DAYS}-day recursive forecast for {EXAMPLE_SCRIP}...")

    # Paths
    scrip_dir = os.path.join(MODELS_DIR, EXAMPLE_SCRIP)
    model_path = os.path.join(scrip_dir, f"lstm_recursive_{EXAMPLE_SCRIP}.keras")
    scaler_path = os.path.join(scrip_dir, f"scaler_{EXAMPLE_SCRIP}.bin")

    if os.path.exists(model_path) and os.path.exists(scaler_path):
        # Load
        model = keras.models.load_model(model_path)
        scaler = joblib.load(scaler_path)

        # Get latest data
        scrip_df = df_all[df_all[SCRIP_COLUMN] == EXAMPLE_SCRIP].sort_values(DATE_COLUMN).reset_index(drop=True)

        # Scale
        full_scaled = scaler.transform(scrip_df[FEATURE_COLS].values)

        # Get last window
        if len(full_scaled) >= SEQ_LEN:
            last_window = full_scaled[-SEQ_LEN:]

            # Recursive Predict
            future_scaled = recursive_predict(model, last_window, PREDICT_DAYS)

            # Inverse
            future_actual = scaler.inverse_transform(future_scaled)

            # Extract Close
            close_idx = FEATURE_COLS.index("Close")
            future_close = future_actual[:, close_idx]

            print("Forecast:", future_close)

            # Plotting
            import matplotlib.pyplot as plt

            # Get recent history for context
            history_days = 60
            history_dates = scrip_df[DATE_COLUMN].iloc[-history_days:].values
            history_close = scrip_df[TARGET_COLUMN].iloc[-history_days:].values

            # Generate future dates
            last_date = pd.to_datetime(history_dates[-1])
            future_dates = [last_date + timedelta(days=i) for i in range(1, PREDICT_DAYS + 1)]

            plt.figure(figsize=(10, 6))
            plt.plot(history_dates, history_close, label='History', color='blue')
            plt.plot(future_dates, future_close, label='Recursive Forecast', color='red', linestyle='--')
            plt.title(f"Recursive Forecast for {EXAMPLE_SCRIP}")
            plt.xlabel("Date")
            plt.ylabel("Close Price")
            plt.legend()
            plt.grid(True)
            plt.show()

        else:
            print("Not enough history for forecast.")
    else:
        print("Model not found.")